In [1]:
import openpmd_api as opmd
import numpy as np
from scipy import constants

from bunchInit_openPMD import vec3D
from bunchInit_openPMD import addParticles2File

# Write

In [ ]:
series = opmd.Series("./test.bp5", opmd.Access.create)
iteration = series.iterations[0]

In [ ]:
# data that is written
# position
x_pos = np.array([0.9])
y_pos = np.array([0.9])
z_pos = np.array([0.9])

# position offset
x_off = np.array([0.0])
y_off = np.array([0.0])
z_off = np.array([0.0])

# extent of the patch (is a hyperrectangle, so 3 dimensions)
patch_x_extent = np.array([1.0])
patch_y_extent = np.array([1.0])
patch_z_extent = np.array([1.0])

# offset of the patch from (0,0,0)
patch_x_offset = np.array([0.0])
patch_y_offset = np.array([0.0])
patch_z_offset = np.array([0.0])

# offset of particle number in the patch
patch_numOffset = np.array([0], dtype=np.uint64)
# number of particles in the patch
patch_numParticles = np.array([1], dtype=np.uint64)

In [ ]:
# get openPMD record
particle = iteration.particles["b_all"]
position = particle["position"]
position_offset = particle["positionOffset"]
particlePatch = particle.particle_patches
patch_offset = particlePatch["offset"]
patch_extent = particlePatch["extent"]

# datasets for new data
dataset_pos = opmd.Dataset(x_pos.dtype, x_pos.shape)
dataset_off = opmd.Dataset(x_off.dtype, x_off.shape)
dataset_patch = opmd.Dataset(patch_x_extent.dtype, patch_x_extent.shape)
dataset_patch_num = opmd.Dataset(np.dtype("uint64"), extent=patch_x_extent.shape)

In [ ]:
# create new datasets in openPMD
position["x"].reset_dataset(dataset_pos)
position["y"].reset_dataset(dataset_pos)
position["z"].reset_dataset(dataset_pos)

position_offset["x"].reset_dataset(dataset_off)
position_offset["y"].reset_dataset(dataset_off)
position_offset["z"].reset_dataset(dataset_off)

patch_offset["x"].reset_dataset(dataset_patch)
patch_offset["y"].reset_dataset(dataset_patch)
patch_offset["z"].reset_dataset(dataset_patch)

patch_extent["x"].reset_dataset(dataset_patch)
patch_extent["y"].reset_dataset(dataset_patch)
patch_extent["z"].reset_dataset(dataset_patch)

particlePatch["numParticlesOffset"][opmd.Mesh_Record_Component.SCALAR].reset_dataset(dataset_patch_num)
particlePatch["numParticles"][opmd.Mesh_Record_Component.SCALAR].reset_dataset(dataset_patch_num)

In [ ]:
# set unit dimensions of patch
# for position, they are set by default
patch_offset.unit_dimension = {
    opmd.Unit_Dimension.L:  1,
}
patch_extent.unit_dimension = {
    opmd.Unit_Dimension.L:  1,
}

# set units
position["x"].unit_SI = 1
position["y"].unit_SI = 1
position["z"].unit_SI = 1

position_offset["x"].unit_SI = 1
position_offset["y"].unit_SI = 1
position_offset["z"].unit_SI = 1

patch_offset["x"].unit_SI = 1
patch_offset["y"].unit_SI = 1
patch_offset["z"].unit_SI = 1

patch_extent["x"].unit_SI = 1
patch_extent["y"].unit_SI = 1
patch_extent["z"].unit_SI = 1

In [ ]:
# write data
position["x"][:] = x_pos
position["y"][:] = y_pos
position["z"][:] = z_pos

position_offset["x"][:] = x_off
position_offset["y"][:] = y_off
position_offset["z"][:] = z_off

patch_offset["x"].store(0, patch_x_offset)
patch_offset["y"].store(0, patch_y_offset)
patch_offset["z"].store(0, patch_z_offset)

patch_extent["x"].store(0, patch_x_extent)
patch_extent["y"].store(0, patch_y_extent)
patch_extent["z"].store(0, patch_z_extent)

particlePatch["numParticlesOffset"][opmd.Mesh_Record_Component.SCALAR].store(0, patch_numOffset)
particlePatch["numParticles"][opmd.Mesh_Record_Component.SCALAR].store(0, patch_numParticles)

In [ ]:
series.flush()
series.close()

# Write (script)

In [2]:
# convert data to 3d vector object
pos = vec3D(np.array([0, 0, 1, 5, 1])*1e-6, np.array([0, 1, 0, 0, 1])*1e-6, np.array([1, 0, 0, 5, 1])*1e-6)
mom = vec3D(np.array([0, 0, 0, 0, 0]), np.array([0, 0, 0, 0, 0]), np.array([0, 0, 0, 0, 0]))
weighting = np.array([7, 8, 9, 10, 11])

In [3]:
# assign to file
# replace with your own paths and species name (from speciesDefinition.param of the input simulation)
particleFile_b = addParticles2File(
    "/p/project1/pwfa-trojan/wrobel1/test6_moreParticles.bp5",
    speciesName="b",
    verbose=True,
)


contains probeE = False
contains probeB False
contains id =  False
contains momentumPrev1 =  False
contains transitionRadiationMask =  False


In [4]:
# write data to file
particleFile_b.addParticles(pos, mom, weighting)
particleFile_b.writeParticles()

# delete data we wrote
del particleFile_b

Make patch mask
0 particles were lost to mask
Writing to file
	Create Series
	Create patch datasets
	Set unit dimensions
	Set units
	Write data
		Write position
		Write position offset
		Write momentum
		Write weighting
		Write patch data
	Flush
Finished


[Warning] Use of group-based encoding in ADIOS2 is discouraged as it can lead
to drastic performance issues, no matter if I/O steps are used or not.

* If not using I/O steps: A crash will corrupt all data since there is only
  one atomic logical write operation upon closing the file.
  Memory performance can be pathological depending on the setup.
* If using I/O steps: Each step will add new variables and attributes instead
  of reusing those from earlier steps. ADIOS2 is not optimized for this and
  especially the BP5 engine will show a quadratic increase in metadata size
  as the number of steps increase. Consider moving to the BP4 engine if you want
  to keep using group-based encoding.
We advise you to pick either file-based encoding or variable-based encoding
(variable-based encoding is not yet feature-complete in the openPMD-api).
For more details, refer to
https://openpmd-api.readthedocs.io/en/latest/usage/concepts.html#iteration-and-series


# Read

In [ ]:
series0 = opmd.Series("./test.bp5", opmd.Access.read_only)
series1 = opmd.Series("/p/scratch/pwfa-trojan/wrobel1/PWFA_Gauss/12_Gauss_filled_lessSuper/simOutput/openPMD/simData_beta_%T.bp5", opmd.Access.read_only)
iteration0 = series0.iterations[0]
iteration1 = series1.iterations[12000]

In [ ]:
a = iteration1.particles["b_all"]["momentum"]
#b = a.load()
series1.flush()

In [ ]:
print(a)
#print(b)
print(a.attributes)
print(a.get_attribute("unitDimension"))

In [ ]:
m_e = constants.m_e
c = constants.c
CELL_WIDTH_SI = 0.5 * 0.1772e-6
CELL_HEIGHT_SI = CELL_WIDTH_SI
CELL_DEPTH_SI = CELL_WIDTH_SI
BASE_DENSITY_SI = 4e24
TYPICAL_PARTICLES_PER_CELL = 1

unit_mass = m_e * CELL_WIDTH_SI * CELL_HEIGHT_SI * CELL_DEPTH_SI * BASE_DENSITY_SI * TYPICAL_PARTICLES_PER_CELL
unit_speed = c

print(unit_mass * unit_speed) 